# Trace Reading Lab

Companion notebook for **Hands-On: Capturing and Reading Your Own Traces**.

What you will do here:

1. Build a small transformer and profile one training step with `torch.profiler` using a `schedule`.
2. Export a Chrome trace and open it in [Perfetto](https://ui.perfetto.dev).
3. Read the `key_averages()` table.
4. Diagnose a **deliberately slow** training loop: find the planted pathologies in the trace *before* reading the solutions cell at the end.

CPU-safe by default — everything runs on a laptop. Cells that use CUDA are guarded by `torch.cuda.is_available()` and simply do more (GPU activity rows) on a GPU box. Dependency: `torch` only.

In [ ]:
import torch
import torch.nn as nn
from torch.profiler import profile, schedule, ProfilerActivity

print("torch:", torch.__version__)
CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if CUDA else "cpu")
print("device:", DEVICE)

ACTIVITIES = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if CUDA else [])
torch.manual_seed(0)

## 1. A small transformer

Small enough to profile in seconds on CPU, structured enough that the trace has recognizable phases (attention, MLP, backward, optimizer).

In [ ]:
def make_model():
    layer = nn.TransformerEncoderLayer(
        d_model=256, nhead=8, dim_feedforward=1024,
        batch_first=True, dropout=0.0)
    return nn.TransformerEncoder(layer, num_layers=4).to(DEVICE)

model = make_model()
opt = torch.optim.AdamW(model.parameters(), lr=1e-4)

x = torch.randn(8, 128, 256, device=DEVICE)   # (batch, seq, d_model)

def train_step(model, opt, x):
    opt.zero_grad(set_to_none=True)
    out = model(x)
    loss = out.pow(2).mean()
    loss.backward()
    opt.step()
    return loss

# warm up once (allocator + any autotuning) outside the profiler
train_step(model, opt, x)
print("params:", sum(p.numel() for p in model.parameters()))

## 2. Profile with a schedule and export a Chrome trace

- `schedule(wait=1, warmup=2, active=3)`: skip 1 step, trace-and-discard 2, record 3.
- `prof.step()` at the end of each iteration drives the schedule — forget it and the trace is empty.
- `on_trace_ready` exports `trace_baseline.json` when the active window completes.

In [ ]:
def export_handler(path):
    def handler(prof):
        prof.export_chrome_trace(path)
        print("wrote", path)
    return handler

with profile(
    activities=ACTIVITIES,
    schedule=schedule(wait=1, warmup=2, active=3, repeat=1),
    record_shapes=True,
    with_stack=True,
    profile_memory=True,
    on_trace_ready=export_handler("trace_baseline.json"),
) as prof:
    for _ in range(8):
        train_step(model, opt, x)
        prof.step()

### Open it in Perfetto

1. Go to **https://ui.perfetto.dev** (the file is parsed locally in your browser, not uploaded).
2. *Open trace file* → `trace_baseline.json` (in Colab: download it from the file browser first).
3. Navigate with **W/S** (zoom) and **A/D** (pan). Find one training step; click an `aten::addmm` slice and inspect its duration, input shapes, and Python stack.
4. Drag-select one full step and read the aggregate table at the bottom.

## 3. The `key_averages()` table

The textual summary of the same data — sorted by self time, grouped by input shape.

In [ ]:
sort_key = "self_cuda_time_total" if CUDA else "self_cpu_time_total"
print(prof.key_averages(group_by_input_shape=True).table(
    sort_by=sort_key, row_limit=15))

In [ ]:
# GPU-only extras: on a CUDA box, also look at the GPU columns and memory stats.
if CUDA:
    print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))
    print(torch.cuda.memory_summary(abbreviated=True))
else:
    print("No CUDA device - skipping GPU summary (this is fine; the lab is CPU-safe).")

## 4. Exercise: the deliberately slow loop

The training loop below computes the same thing as `train_step`, but it has **three planted performance pathologies**.

Run it, open `trace_slow.json` in Perfetto next to `trace_baseline.json`, and answer:

1. What are the three pathologies? Point at trace evidence for each (op names, slice patterns, per-step time).
2. Which one would hurt *most* on a GPU, and why does the CPU trace understate it?
3. Roughly how much slower is a slow step than a baseline step?

Read the code, but make your case from the trace. Solutions are in the last cell — no peeking until you have three answers.

In [ ]:
model_s = make_model()
opt_s = torch.optim.AdamW(model_s.parameters(), lr=1e-4)

full_batch = torch.randn(8, 128, 256, device=DEVICE)

def slow_train_step(model, opt, batch):
    losses = []
    # Pathology-laden version of train_step. Same math, worse everything.
    for i in range(batch.shape[0]):            # <- ???
        micro = batch[i:i+1]
        opt.zero_grad(set_to_none=True)
        out = model(micro)
        out_host = out.cpu()                    # <- ???
        loss = (out_host.to(DEVICE) if CUDA else out_host).pow(2).mean()
        loss.backward()
        opt.step()
        losses.append(loss.item())              # <- ???
    return sum(losses) / len(losses)

with profile(
    activities=ACTIVITIES,
    schedule=schedule(wait=1, warmup=2, active=3, repeat=1),
    record_shapes=True,
    with_stack=True,
    on_trace_ready=export_handler("trace_slow.json"),
) as prof_slow:
    for _ in range(8):
        slow_train_step(model_s, opt_s, full_batch)
        prof_slow.step()

print(prof_slow.key_averages().table(sort_by=sort_key, row_limit=15))

In [ ]:
# Quick wall-clock comparison to anchor your trace reading.
import time

def timeit(fn, n=5):
    if CUDA:
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n):
        fn()
    if CUDA:
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n

base = timeit(lambda: train_step(model, opt, x))
slow = timeit(lambda: slow_train_step(model_s, opt_s, full_batch))
print(f"baseline step: {base*1e3:8.1f} ms")
print(f"slow step:     {slow*1e3:8.1f} ms   ({slow/base:.1f}x slower)")

---

## 5. Solutions

*Only read this after writing down your three answers.*

**Pathology 1 — tiny batches (`for i in range(batch.shape[0])` with `batch[i:i+1]`).**
The batch of 8 is split into 8 batch-1 micro-steps, so every op launches 8x with shapes too small to use the hardware. Trace evidence: 8 repeated forward/backward patterns per step instead of one; `record_shapes` shows `[1, 128, 256]` inputs; `key_averages` shows call counts multiplied by 8 with low time-per-call. On a GPU this is worse than on CPU: each launch is a kernel that cannot fill the SMs, and launch overhead becomes a first-order cost.

**Pathology 2 — unnecessary host transfer (`out.cpu()` then back).**
A round-trip through host memory in the middle of the step. Trace evidence: `aten::to` / `aten::copy_` (and on GPU, `Memcpy DtoH` + `Memcpy HtoD`) between forward and backward in every micro-step. On a GPU this also *synchronizes*: the CPU must wait for the forward to finish before copying, killing CPU/GPU overlap. The CPU trace understates this — on CPU it is just a copy; on GPU it is a copy plus a pipeline stall.

**Pathology 3 — `.item()` every micro-step.**
`loss.item()` forces a device-to-host sync each iteration, 8 per step. Trace evidence on GPU: `cudaStreamSynchronize` / `cudaMemcpyAsync` in the CUDA API row right after `opt.step()`, with the CPU thread lockstepped to the GPU instead of running ahead. Fix: accumulate losses as tensors and call `.item()` once per logging interval, not per micro-batch.

**The general lesson:** none of these change the math, all of them change the timeline. Batch work up, keep tensors on-device, and synchronize on your schedule, not the loop's.